In [ ]:
from google.colab import drive
import os

def _drive_ready(mountpoint='/content/drive'):
    """진짜로 Drive 가 붙어 있는가.

    isdir('/content/drive/MyDrive') 만 보면 안 된다 — 마운트가 끊긴 자리에 같은 이름의
    폴더가 남으면 붙지도 않았는데 붙은 것처럼 보이고, 그 뒤 읽기·쓰기가 전부 조용히
    로컬 디스크로 새 나간다(한참 뒤 엉뚱한 데서 FileNotFoundError 로 터진다)."""
    return os.path.ismount(mountpoint) and any(
        os.path.isdir(os.path.join(mountpoint, n)) for n in ('MyDrive', 'My Drive')
    )


def _ensure_drive(mountpoint='/content/drive'):
    """이미 붙어 있으면 넘어가고, 아니면 붙인다. 마운트 자리에 찌꺼기가 남아
    mount 가 거부하는 경우("Mountpoint must not already contain files")까지 복구한다."""
    if _drive_ready(mountpoint):
        return
    try:
        drive.mount(mountpoint)
        return
    except ValueError:
        pass

    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    if not os.path.ismount(mountpoint):
        # 마운트가 아니니 여기 있는 건 Drive 원본이 아니라 로컬 찌꺼기다. 지우지 말고
        # 옆으로 치워 둔다 — 이 상태에서 저장된 파일이 있었다면 살릴 수 있게.
        # (rm -rf 로 지우는 방법은 쓰지 않는다. 마운트가 살아 있었다면 원본이 날아간다.)
        stale, n = mountpoint + '_stale', 1
        while os.path.exists(stale):
            n += 1
            stale = f'{mountpoint}_stale{n}'
        os.rename(mountpoint, stale)
        print(f'마운트가 아닌 찌꺼기 폴더를 {stale} 로 치웠습니다.')
    drive.mount(mountpoint, force_remount=True)


_ensure_drive()

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os
import re
import random
import unicodedata

# ============================================================
# 0. PBP 파일명 패턴
# ============================================================
PBP_PATTERN = re.compile(r'^\d{4}_\d{2}_[a-z]+\.xlsx$')


# ============================================================
# 1. IP 변환
# ============================================================
def parse_ip(ip_str):
    """
    KBSA 이닝 표기 변환
    '4' → 4.0, '4.1' → 4.333, '4.2' → 4.667
    """
    try:
        ip_str = str(ip_str).strip()
        if '.' in ip_str:
            full, partial = ip_str.split('.')
            return int(full) + int(partial) / 3
        else:
            return float(ip_str)
    except:
        return None


# ============================================================
# 2. game_idx 수집
# ============================================================
def get_game_idx_from_pbp_folders(base_dir, years=None):
    """
    연도별/월별 폴더에서 game_id 추출
    예: base_dir/2021/6월/2021_06_a.xlsx

    years: None이면 전체, 리스트로 지정하면 해당 연도만
    예: years=[2022] 또는 years=[2021, 2022]
    """
    if years is None:
        years = range(2020, 2027)

    game_idx_by_year = {}

    for year in years:
        year_dir = os.path.join(base_dir, str(year))
        if not os.path.exists(year_dir):
            print(f"{year}년 폴더 없음, 스킵")
            continue

        ids = []

        for root, dirs, files in os.walk(year_dir):
            dirs[:] = [unicodedata.normalize('NFC', d) for d in dirs]

            for fname in files:
                fname = unicodedata.normalize('NFC', fname)

                # 임시 파일 스킵
                if fname.startswith('~$'):
                    continue
                # PBP 파일 패턴만 허용
                if not PBP_PATTERN.match(fname):
                    continue

                fpath = os.path.join(root, fname)
                try:
                    df = pd.read_excel(fpath, usecols=['game_id'])
                    game_ids = df['game_id'].dropna().astype(int).unique().tolist()
                    ids.extend(game_ids)
                except Exception as e:
                    print(f"  [{fname}] 읽기 실패: {e}")
                    continue

        ids = sorted(set(ids))
        game_idx_by_year[year] = ids
        print(f"{year}년: {len(ids)}개 game_idx 수집")

    return game_idx_by_year


# ============================================================
# 3. 페이지 파싱
# ============================================================
def parse_pitcher_box(game_idx, session=None):
    url = f"https://www.korea-baseball.com/game/record_detail?game_idx={game_idx}"

    try:
        if session:
            res = session.get(url, timeout=10)
        else:
            res = requests.get(url, timeout=10)
        res.raise_for_status()
    except Exception as e:
        print(f"  [{game_idx}] 요청 실패: {e}")
        return []

    soup = BeautifulSoup(res.text, 'html.parser')
    records = []

    # team1, team2
    hero = soup.select('.hero_sum .team')
    team1 = hero[0].select_one('.txt').get_text(strip=True) if len(hero) > 0 else None
    team2 = hero[1].select_one('.txt').get_text(strip=True) if len(hero) > 1 else None

    # 투수기록 섹션 (팀별 2개)
    sections = soup.select('.section_sumrec')

    for section in sections:
        h4 = section.select_one('h4')
        if not h4 or '투수기록' not in h4.get_text():
            continue

        def_team = h4.get_text(strip=True).replace('투수기록', '').strip()

        table = section.select_one('table[summary="투수기록"]')
        if not table:
            continue

        rows = table.select('tbody tr')
        for row in rows:
            if 'sum' in row.get('class', []):
                continue

            cells = row.find_all(['th', 'td'])
            if len(cells) < 16:
                continue

            try:
                record = {
                    'game_idx': game_idx,
                    'team1':    team1,
                    'team2':    team2,
                    'def_team': def_team,
                    'pitcher':  cells[0].get_text(strip=True),
                    'entry':    cells[1].get_text(strip=True),
                    'result':   cells[2].get_text(strip=True),
                    'W':        cells[3].get_text(strip=True),
                    'L':        cells[4].get_text(strip=True),
                    'IP_raw':   cells[5].get_text(strip=True),
                    'BF':       cells[6].get_text(strip=True),
                    'pitches':  cells[7].get_text(strip=True),
                    'AB':       cells[8].get_text(strip=True),
                    'H':        cells[9].get_text(strip=True),
                    'HR':       cells[10].get_text(strip=True),
                    'BB_HBP':   cells[11].get_text(strip=True),
                    'K':        cells[12].get_text(strip=True),
                    'R':        cells[13].get_text(strip=True),
                    'ER':       cells[14].get_text(strip=True),
                    'ERA':      cells[15].get_text(strip=True),
                }
                records.append(record)
            except Exception as e:
                print(f"  [{game_idx}] 행 파싱 오류: {e}")
                continue

    return records


# ============================================================
# 4. 저장
# ============================================================
def _save_pitcher_box(records, output_path):
    if not records:
        return pd.DataFrame()

    df = pd.DataFrame(records)

    # IP 변환 (raw 보존 + float 컬럼 추가)
    df['IP'] = df['IP_raw'].apply(parse_ip)

    # 정수 컬럼
    int_cols = ['W', 'L', 'BF', 'pitches', 'AB', 'H', 'HR', 'BB_HBP', 'K', 'R', 'ER']
    for col in int_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

    # ERA
    df['ERA'] = pd.to_numeric(df['ERA'], errors='coerce')

    # 컬럼 순서
    col_order = ['game_idx', 'team1', 'team2', 'def_team', 'pitcher',
                 'entry', 'result', 'W', 'L', 'IP_raw', 'IP',
                 'BF', 'pitches', 'AB', 'H', 'HR', 'BB_HBP', 'K', 'R', 'ER', 'ERA']
    df = df[[c for c in col_order if c in df.columns]]

    df.to_excel(output_path, index=False)
    return df


# ============================================================
# 5. 메인 크롤링
# ============================================================
def crawl_pitcher_box_by_year(game_idx_by_year, output_dir, save_every=50, max_retry=3):
    """
    game_idx_by_year : {2021: [id1, id2, ...], ...}
    output_dir       : 저장 폴더
    save_every       : N경기마다 중간 저장
    max_retry        : 실패 시 최대 재시도 횟수
    """
    os.makedirs(output_dir, exist_ok=True)

    session = requests.Session()
    session.headers.update({
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    })

    total_failed = {}

    for year, game_idx_list in sorted(game_idx_by_year.items()):
        print(f"\n{'='*50}")
        print(f"{year}년 수집 시작 ({len(game_idx_list)}경기)")
        print(f"{'='*50}")

        all_records = []
        failed = []
        output_path = os.path.join(output_dir, f'pitcher_box_{year}.xlsx')

        for i, game_idx in enumerate(game_idx_list):

            # 재시도 로직
            records = []
            elapsed = 0
            for attempt in range(1, max_retry + 1):
                start = time.time()
                records = parse_pitcher_box(game_idx, session)
                elapsed = time.time() - start

                if records:
                    print(f"  [{i+1}/{len(game_idx_list)}] {game_idx} → {len(records)}명 ({elapsed:.1f}s)")
                    break
                else:
                    if attempt < max_retry:
                        retry_wait = random.uniform(9.0, 11.0)
                        print(f"  [{i+1}/{len(game_idx_list)}] {game_idx} → 실패 ({elapsed:.1f}s) / 재시도 {attempt}/{max_retry} ({retry_wait:.1f}s 후)")
                        time.sleep(retry_wait)
                    else:
                        print(f"  [{i+1}/{len(game_idx_list)}] {game_idx} → 최종 실패 ({max_retry}회 시도)")
                        failed.append(game_idx)

            if records:
                all_records.extend(records)

            # 중간 저장
            if (i + 1) % save_every == 0:
                _save_pitcher_box(all_records, output_path)
                print(f"  >>> 중간 저장 완료 ({i+1}/{len(game_idx_list)})")

            # 응답시간 포함 9~11초 랜덤 간격 유지
            target = random.uniform(9.0, 11.0)
            wait = max(0, target - elapsed)
            time.sleep(wait)

        # 최종 저장
        df = _save_pitcher_box(all_records, output_path)
        total_failed[year] = failed

        print(f"\n{year}년 완료: {len(df)}행 저장")
        print(f"실패 {len(failed)}개: {failed if failed else '없음'}")

    return total_failed


# ============================================================
# 실행
# ============================================================
BASE_DIR = '/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)'
OUT_DIR  = os.path.join(BASE_DIR, 'pitcher_box')

# game_idx 수집
# 전체: years=None
# 특정 연도: years=[2022] 또는 years=[2021, 2022]
game_idx_by_year = get_game_idx_from_pbp_folders(BASE_DIR, years=[2022])

# 수집 현황 확인
print("\n[수집 현황]")
for year, ids in sorted(game_idx_by_year.items()):
    print(f"  {year}년: {len(ids)}경기")

# 크롤링 시작
failed = crawl_pitcher_box_by_year(
    game_idx_by_year,
    output_dir=OUT_DIR,
    save_every=50,
    max_retry=3,
)

# 최종 실패 목록
print("\n[최종 실패 목록]")
for year, ids in failed.items():
    if ids:
        print(f"  {year}년: {ids}")

In [ ]:
import re

# 패턴: 연도_월_알파벳(들).xlsx
PBP_PATTERN = re.compile(r'^\d{4}_\d{2}_[a-z]+\.xlsx$')

# 필터 예시
for fname in ['2021_08_j.xlsx', '2022_player_stats_plus.xlsx', '2023_pitcher_stats.xlsx', '2021_08_aa.xlsx']:
    print(fname, '->', bool(PBP_PATTERN.match(fname)))